We first need to download Geocoder

In [ ]:
import sys
!{sys.executable} -m pip install pandas tqdm


   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ------------- -------------------------- 1/3 [geographiclib]
   -------------------------- ------------- 2/3 [geopy]
   -------------------------- ------------- 2/3 [geopy]
   -------------------------- ------------- 2/3 [geopy]
   -------------------------- ------------- 2/3 [geopy]
   -------------------------- ------------- 2/3 [geopy]
   -------------------------- ------------- 2/3 [geopy]
   ---------------------------------------- 3/3 [geopy]



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Libraries

In [22]:
import pandas as pd
import os


In [12]:
import pandas as pd

df = pd.read_csv("ExportBuildings.csv")

df.head()

,Building_Name,Campus,Address
0,110 SOUTH ADAMS STREET,MONROE PARK CAMPUS,110 SOUTH ADAMS STREET
1,1109 WEST MARSHALL ST,MONROE PARK CAMPUS,1109 WEST MARSHALL ST
2,1235 W. BROAD ST RETAIL 1,MONROE PARK CAMPUS,1235 W. BROAD ST RETAIL 1
3,"125 S. HARRISON, SOCCER LOCKER ROOM",MONROE PARK CAMPUS,125 SOUTH HARRISON STREET
4,1310-1312 WEST MAIN ST,MONROE PARK CAMPUS,1310-1312 WEST MAIN STREET


Checking the columns

In [13]:
df.columns

Index(['Building_Name', 'Campus', 'Address'], dtype='str')

Looking for missings to fix

In [14]:
df[["Building_Name", "Campus", "Address"]].head(15)

,Building_Name,Campus,Address
0,110 SOUTH ADAMS STREET,MONROE PARK CAMPUS,110 SOUTH ADAMS STREET
1,1109 WEST MARSHALL ST,MONROE PARK CAMPUS,1109 WEST MARSHALL ST
2,1235 W. BROAD ST RETAIL 1,MONROE PARK CAMPUS,1235 W. BROAD ST RETAIL 1
3,"125 S. HARRISON, SOCCER LOCKER ROOM",MONROE PARK CAMPUS,125 SOUTH HARRISON STREET
4,1310-1312 WEST MAIN ST,MONROE PARK CAMPUS,1310-1312 WEST MAIN STREET
5,1607 SHERWOOD AVE,OFF-CAMPUS,1607 SHERWOOD AVE
6,1609 SHERWOOD AVE,OFF-CAMPUS,1609 SHERWOOD AVE
7,201 N. BELVIDERE STREET,MONROE PARK CAMPUS,201 NORTH BELVIDERE STREET
8,205 E. BROAD ST,0,205 E. BROAD ST
9,205 E. BROAD ST,MONROE PARK CAMPUS,205 E. BROAD ST


Missing addresses

In [16]:
missing_addr = df[df["Address"].isna() | (df["Address"].astype(str).str.strip() == "")]

missing_addr.head(30)


,Building_Name,Campus,Address
13,2303 N. PARHAM ROAD,OFF-CAMPUS,NaN
15,2601 HERMITAGE ROAD,MONROE PARK CAMPUS,NaN
19,2901 HERMITAGE ROAD,OFF-CAMPUS,NaN
42,ADULT OUTPATIENT (AOP),MCV CAMPUS,NaN
50,BIOTECH CENTER,NaN,NaN
61,CABANISS HALL,NaN,NaN
76,CLINICAL SUPPORT CENTER,NaN,NaN
79,COMMONWEALTH OF VIRGINIA (STEAM),NaN,NaN
80,COMMONWEALTH OF VIRGINIA (STEAM),OFF-CAMPUS,NaN
82,CRITICAL CARE HOSPITAL,NaN,NaN


Unique buildings

In [ ]:
bldgs = (
    df[["Building_Name", "Campus"]]
    .dropna(subset=["Building_Name"])
    .assign(Building_Name=lambda x: x["Building_Name"].astype(str).str.strip())
    .query("Building_Name != ''")
    .drop_duplicates(subset=["Building_Name"])
    .reset_index(drop=True)
)

bldgs.shape, bldgs.head(10)

bldgs.to_csv("vcu_unique_buildings.csv", index = False)

print("Export complete: vcu_unique_buildings.csv")

((255, 2),
                          Building_Name              Campus
 0               110 SOUTH ADAMS STREET  MONROE PARK CAMPUS
 1                1109 WEST MARSHALL ST  MONROE PARK CAMPUS
 2            1235 W. BROAD ST RETAIL 1  MONROE PARK CAMPUS
 3  125 S. HARRISON, SOCCER LOCKER ROOM  MONROE PARK CAMPUS
 4               1310-1312 WEST MAIN ST  MONROE PARK CAMPUS
 5                    1607 SHERWOOD AVE          OFF-CAMPUS
 6                    1609 SHERWOOD AVE          OFF-CAMPUS
 7              201 N. BELVIDERE STREET  MONROE PARK CAMPUS
 8                      205 E. BROAD ST                   0
 9          205 W. FRANKLIN ST #3, 3 FL  MONROE PARK CAMPUS)

Search Query to change the address

In [19]:

def make_query(row):
    name = row["Building_Name"]
    campus = str(row.get("Campus", "")).strip().lower()

    if "mcv" in campus:
        campus_hint = "VCU Medical Campus"
    elif "monroe" in campus:
        campus_hint = "VCU Monroe Park Campus"
    else:
        campus_hint = "Virginia Commonwealth University"
    
    return f"{name}, {campus_hint}, Richmond, VA"

bldgs["Query"] = bldgs.apply(make_query, axis=1)
bldgs[["Building_Name", "Campus", "Query"]].head(15)

,Building_Name,Campus,Query
0,110 SOUTH ADAMS STREET,MONROE PARK CAMPUS,"110 SOUTH ADAMS STREET, VCU Monroe Park Campus..."
1,1109 WEST MARSHALL ST,MONROE PARK CAMPUS,"1109 WEST MARSHALL ST, VCU Monroe Park Campus,..."
2,1235 W. BROAD ST RETAIL 1,MONROE PARK CAMPUS,"1235 W. BROAD ST RETAIL 1, VCU Monroe Park Cam..."
3,"125 S. HARRISON, SOCCER LOCKER ROOM",MONROE PARK CAMPUS,"125 S. HARRISON, SOCCER LOCKER ROOM, VCU Monro..."
4,1310-1312 WEST MAIN ST,MONROE PARK CAMPUS,"1310-1312 WEST MAIN ST, VCU Monroe Park Campus..."
5,1607 SHERWOOD AVE,OFF-CAMPUS,"1607 SHERWOOD AVE, Virginia Commonwealth Unive..."
6,1609 SHERWOOD AVE,OFF-CAMPUS,"1609 SHERWOOD AVE, Virginia Commonwealth Unive..."
7,201 N. BELVIDERE STREET,MONROE PARK CAMPUS,"201 N. BELVIDERE STREET, VCU Monroe Park Campu..."
8,205 E. BROAD ST,0,"205 E. BROAD ST, Virginia Commonwealth Univers..."
9,"205 W. FRANKLIN ST #3, 3 FL",MONROE PARK CAMPUS,"205 W. FRANKLIN ST #3, 3 FL, VCU Monroe Park C..."
